In [1]:
import warnings
warnings.filterwarnings('ignore')

!pip uninstall -y cupy-cuda11x cupy-cuda12x spacy
!pip install -U spacy[cuda12x] pandas scikit-learn
!python -m spacy download en_core_web_sm

Found existing installation: cupy-cuda12x 12.3.0
Uninstalling cupy-cuda12x-12.3.0:
  Successfully uninstalled cupy-cuda12x-12.3.0
Found existing installation: spacy 3.8.11
Uninstalling spacy-3.8.11:
  Successfully uninstalled spacy-3.8.11
  Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (27 kB)
  Using cached cupy_cuda12x-12.3.0-cp312-cp312-manylinux2014_x86_64.whl.metadata (2.7 kB)
Using cached cupy_cuda12x-12.3.0-cp312-cp312-manylinux2014_x86_64.whl (82.1 MB)
Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, but you have cupy-cuda12x 12.3.0 which is incompatible.
cudf-cu12 26.2.1 requires cupy-cuda12x>=13.6.0, but you have cupy-cuda12x 12.3.0 which is incompatible.
c

In [ ]:
import spacy

try:
    spacy.require_gpu()
    print("✅ Using GPU")
except:
    print("⚠️ GPU not available, using CPU")

✅ Using GPU


In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
from google.colab import files
uploaded = files.upload()

Saving Lables.csv to Lables.csv


In [5]:
import pandas as pd

df = pd.read_csv("/content/Lables.csv")
df.head()

,Id,UserId,MessageContent,Time,Sender,MessageId,Type,Amount,Target,Alias,Account
0,b79bbf7c-6e8b-4356-929b-026c8277e390,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-26 11:44:18+00,AX-ICICIT-S,cc53375a-447d-4b42-a819-7e1c9f6db10b,Debit,80.00,AWFIS,5.90E+11,ICICI:XX789
1,ec6680c2-6aa9-4675-8ac2-7cd1f4820a76,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 306.21 on...,2025-12-26 15:24:11+00,AX-ICICIT-S,3cab480f-410f-4482-a685-afb1db248290,Debit,306.21,ZOMATO,9.90E+11,ICICI:XX789
2,e9fb8ada-2666-401e-b7be-152c9a9c7fdd,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Credited for Rs:5000.00 on 26-12-202...,2025-12-26 15:41:35+00,JD-UNIONB-S,c82bf5ba-4931-4fd3-b711-ff40c1a36c1c,Credit,5000.00,Unknown,Unknown,UNION:*0186
3,80da7ab7-8b35-4132-8e33-11a22f124dbf,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Debited for Rs:600.00 on 28-12-2025 ...,2025-12-28 15:01:24+00,JD-UNIONB-S,f645a1aa-386c-4872-a25b-569469cabcb7,Debit,600.00,Unknown,Unknown,UNION:*0186
4,d290bf24-3959-4cb8-95ab-2b1309b3955c,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-28 15:21:45+00,AX-ICICIT-S,58f58044-fe67-44f6-9465-23d9938727c4,Debit,80.00,MRS GEDDA RAMA,7.71E+11,ICICI:XX789


In [6]:
df["MessageContent"] = df["MessageContent"].astype(str)
df["Sender"] = df["Sender"].astype(str)

# Normalize
df["Target"] = df["Target"].astype(str).str.upper()
df["Account"] = df["Account"].astype(str).str.upper()
df["Type"] = df["Type"].astype(str).str.upper()

In [7]:
def create_training_data(df):
    TRAIN_DATA = []

    for _, row in df.iterrows():
        text = str(row["MessageContent"])
        text_upper = text.upper()
        entities = []

        # -------- ACCOUNT --------
        def add_account(value):
            if pd.isna(value):
                return

            value = str(value)

            if ":" in value:
                value = value.split(":")[1]  # remove ICICI:

            start = text_upper.find(value.upper())
            if start != -1:
                entities.append((start, start + len(value), "ACCOUNT"))

        # -------- TARGET --------
        def add_entity(value, label):
            if pd.isna(value):
                return

            value = str(value).strip()
            if value.lower() == "unknown":
                return

            start = text_upper.find(value.upper())
            if start != -1:
                entities.append((start, start + len(value), label))

        # -------- AMOUNT --------
        def add_amount(amount):
            if pd.isna(amount):
                return

            amount = str(amount)

            patterns = [
                f"RS {amount}",
                f"RS.{amount}",
                f"RS {amount}.00",
                f"{amount}.00"
            ]

            for p in patterns:
                start = text_upper.find(p.upper())
                if start != -1:
                    entities.append((start, start + len(p), "AMOUNT"))
                    return

        # -------- TYPE --------
        def add_type(text):
            if "DEBITED" in text:
                start = text.find("DEBITED")
                entities.append((start, start + 7, "TYPE"))
            elif "CREDITED" in text:
                start = text.find("CREDITED")
                entities.append((start, start + 8, "TYPE"))

        # -------- BANK --------
        def add_bank(text):
            if "ICICI" in text:
                start = text.find("ICICI")
                entities.append((start, start + 5, "BANK"))
            elif "HDFC" in text:
                start = text.find("HDFC")
                entities.append((start, start + 4, "BANK"))
            elif "SBI" in text:
                start = text.find("SBI")
                entities.append((start, start + 3, "BANK"))
            elif "AXIS" in text:
                start = text.find("AXIS")
                entities.append((start, start + 4, "BANK"))
            elif "UNION BANK" in text:
                start = text.find("UNION BANK")
                entities.append((start, start + 11, "BANK"))

        # Apply all
        add_account(row["Account"])
        add_entity(row["Target"], "TARGET")
        add_amount(row["Amount"])
        add_type(text_upper)
        add_bank(text_upper)

        if entities:
            TRAIN_DATA.append((text, {"entities": entities}))

    return TRAIN_DATA


TRAIN_DATA = create_training_data(df)

print("Samples:", len(TRAIN_DATA))
print(TRAIN_DATA[:2])

Samples: 653
[('ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; AWFIS credited. UPI:590283159319. Call 18002662 for dispute. SMS BLOCK 789 to 9215676766.', {'entities': [(16, 21, 'ACCOUNT'), (57, 62, 'TARGET'), (34, 41, 'AMOUNT'), (22, 29, 'TYPE'), (0, 5, 'BANK')]}), ('ICICI Bank Acct XX789 debited for Rs 306.21 on 26-Dec-25; ZOMATO credited. UPI:989788759216. Call 18002662 for dispute. SMS BLOCK 789 to 9215676766.', {'entities': [(16, 21, 'ACCOUNT'), (58, 64, 'TARGET'), (34, 43, 'AMOUNT'), (22, 29, 'TYPE'), (0, 5, 'BANK')]})]


In [ ]:
import spacy
from spacy.training.example import Example
import random

nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

labels = ["AMOUNT", "ACCOUNT", "TARGET", "TYPE", "BANK"]

for label in labels:
    ner.add_label(label)

optimizer = nlp.begin_training()

for epoch in range(20):
    random.shuffle(TRAIN_DATA)
    losses = {}

    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], losses=losses)

    print(f"Epoch {epoch}: {losses}")

Epoch 0: {'ner': 557.3664414832965}
Epoch 1: {'ner': 1.9232598624676625}
Epoch 2: {'ner': 7.846012287810852e-05}
Epoch 3: {'ner': 156.42979400646678}
Epoch 4: {'ner': 103.75130740793905}
Epoch 5: {'ner': 2.423218868611987}
Epoch 6: {'ner': 5.671784259452159e-06}
Epoch 7: {'ner': 3.939535057968968e-06}
Epoch 8: {'ner': 2.7158094713690715e-07}
Epoch 9: {'ner': 1.6537013874792213e-07}
Epoch 10: {'ner': 5.839580971116961e-07}
Epoch 11: {'ner': 4.411950407914412e-08}
Epoch 12: {'ner': 1.5618637945209548e-08}
Epoch 13: {'ner': 3.590305996413488e-09}
Epoch 14: {'ner': 1.4151475210215321e-08}
Epoch 15: {'ner': 2.3850403700884418e-08}
Epoch 16: {'ner': 1.2600374196739051e-09}
Epoch 17: {'ner': 5.055482403761168e-06}
Epoch 18: {'ner': 1.9410975229538628e-09}
Epoch 19: {'ner': 4.488189521354472e-10}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

alias_df = df[df["Alias"].notna()]
alias_df = alias_df[alias_df["Alias"] != "Unknown"]

alias_df["input_text"] = (
    alias_df["Sender"] + " " + alias_df["MessageContent"]
)

X = alias_df["input_text"]
y = alias_df["Alias"]

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,3),
    min_df=2
)

X_vec = vectorizer.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, y)

print("Alias model trained")

Alias model trained


In [ ]:
import re

def predict_alias(sender, sms):
    text = sender + " " + sms
    vec = vectorizer.transform([text])

    probs = model.predict_proba(vec)
    confidence = probs.max()

    if confidence < 0.6:
        return "Unknown"

    return model.classes_[probs.argmax()]


def parse_amount(amount_text):
    """Parse amount text into structured format: {currency, value}"""
    if not amount_text:
        return None
    
    amount_text = str(amount_text).strip()
    
    # Extract numeric value
    numeric_match = re.search(r'\d+\.?\d*', amount_text)
    if not numeric_match:
        return None
    
    value = float(numeric_match.group())
    
    # Identify currency
    currency_map = {
        'rs': 'Rupee',
        'r': 'Rupee',
        '₹': 'Rupee',
        'inr': 'Rupee'
    }
    
    currency = 'Rupee'  # Default
    for curr_key, curr_name in currency_map.items():
        if curr_key.lower() in amount_text.lower():
            currency = curr_name
            break
    
    return {
        "currency": currency,
        "value": value
    }


def parse_sms(sender, sms):
    doc = nlp(sms)

    result = {
        "Type": None,
        "Amount": None,
        "Target": None,
        "Account": None,
        "Alias": None
    }

    bank = None
    account = None

    for ent in doc.ents:
        if ent.label_ == "TYPE":
            result["Type"] = ent.text
        elif ent.label_ == "AMOUNT":
            result["Amount"] = parse_amount(ent.text)
        elif ent.label_ == "TARGET":
            result["Target"] = ent.text
        elif ent.label_ == "ACCOUNT":
            account = ent.text
        elif ent.label_ == "BANK":
            bank = ent.text.upper()

    if bank and account:
        result["Account"] = f"{bank}:{account}"
    else:
        result["Account"] = account

    result["Alias"] = predict_alias(sender, sms)

    return result

In [11]:
sms = "ICICI Bank Acct XX789 debited for Rs 30.00 on 30-Dec-25; RAJESH REDDY MU credited. UPI:552084545580"
sender = "AD-ICICIT-S"

print(parse_sms(sender, sms))

{'Type': 'debited', 'Amount': 'Rs 30.00', 'Target': 'RAJESH REDDY MU', 'Account': 'ICICI:XX789', 'Alias': 'Unknown'}


In [12]:
import pickle
import shutil
import os

# Save NER
nlp.to_disk("sms_ner_model")

# Save alias model
pickle.dump(model, open("alias_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

# Create a temporary directory for models
models_dir = "./temp_models_to_zip"
os.makedirs(models_dir, exist_ok=True)

# Move model files to the temporary directory
shutil.move("sms_ner_model", models_dir)
shutil.move("alias_model.pkl", models_dir)
shutil.move("vectorizer.pkl", models_dir)

# Zip the temporary directory
shutil.make_archive("models", 'zip', models_dir)

# Clean up the temporary directory
shutil.rmtree(models_dir)


In [13]:
from google.colab import files
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>